In [1]:
import pandas as pd
import os

# 1. Load the CSV files
# Note: '../data/' assumes your notebook is in 'python/notebooks/'
df_2m = pd.read_csv('../data/EURUSD_2m.csv', parse_dates=['time'])
df_15m = pd.read_csv('../data/EURUSD_15m.csv', parse_dates=['time'])
df_4h = pd.read_csv('../data/EURUSD_4h.csv', parse_dates=['time'])

# 2. Sort by time (required for merge_asof)
df_2m = df_2m.sort_values('time')
df_15m = df_15m.sort_values('time')
df_4h = df_4h.sort_values('time')

# 3. Merge 15m data into the 2m base
# direction='backward' means: "For this 2m candle, give me the most recent 15m candle that has closed"
merged = pd.merge_asof(df_2m, df_15m, on='time', direction='backward', suffixes=('', '_15m'))

# 4. Merge 4h data into the result
final_df = pd.merge_asof(merged, df_4h, on='time', direction='backward', suffixes=('', '_4h'))

# 5. Review the result
print(f"Total Rows: {len(final_df)}")
print("\nColumns available now:")
print(final_df.columns.tolist())

# Show the first few rows to verify the merge
final_df.head()

Total Rows: 5000

Columns available now:
['time', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume', 'open_15m', 'high_15m', 'low_15m', 'close_15m', 'tick_volume_15m', 'spread_15m', 'real_volume_15m', 'open_4h', 'high_4h', 'low_4h', 'close_4h', 'tick_volume_4h', 'spread_4h', 'real_volume_4h']


,time,open,high,low,close,tick_volume,spread,real_volume,open_15m,high_15m,...,tick_volume_15m,spread_15m,real_volume_15m,open_4h,high_4h,low_4h,close_4h,tick_volume_4h,spread_4h,real_volume_4h
0,2026-04-07 11:28:00,1.15563,1.15564,1.15530,1.15537,81,8,0,1.15487,1.15564,...,696,8,0,1.15419,1.15753,1.15416,1.15597,15003,8,0
1,2026-04-07 11:30:00,1.15537,1.15537,1.15517,1.15521,79,8,0,1.15537,1.15617,...,706,8,0,1.15419,1.15753,1.15416,1.15597,15003,8,0
2,2026-04-07 11:32:00,1.15521,1.15525,1.15504,1.15516,60,8,0,1.15537,1.15617,...,706,8,0,1.15419,1.15753,1.15416,1.15597,15003,8,0
3,2026-04-07 11:34:00,1.15516,1.15540,1.15516,1.15539,48,8,0,1.15537,1.15617,...,706,8,0,1.15419,1.15753,1.15416,1.15597,15003,8,0
4,2026-04-07 11:36:00,1.15539,1.15592,1.15520,1.15571,133,8,0,1.15537,1.15617,...,706,8,0,1.15419,1.15753,1.15416,1.15597,15003,8,0


In [2]:
# 1. Technical Indicators for the 15m timeframe
# Calculate a 20-period Moving Average on the 15m data
final_df['sma_15m'] = final_df['close_15m'].rolling(window=20).mean()

# Feature: Distance from SMA (tells the AI if price is overextended)
final_df['dist_from_sma_15m'] = final_df['close'] - final_df['sma_15m']

# 2. 4H Trend Direction
# 1 if 4H candle is bullish, -1 if bearish
final_df['trend_4h'] = (final_df['close_4h'] > final_df['open_4h']).astype(int).replace(0, -1)

# 3. Target Variable (The "Answer Key" for the AI)
# Let's predict if price will be higher in 10 minutes (5 bars of 2min)
final_df['target'] = (final_df['close'].shift(-5) > final_df['close']).astype(int)

# 4. Clean up
final_df.dropna(inplace=True)

print("Features Created!")
final_df[['time', 'close', 'dist_from_sma_15m', 'trend_4h', 'target']].tail(10)

Features Created!


,time,close,dist_from_sma_15m,trend_4h,target
4990,2026-04-16 10:30:00,1.17739,-0.000329,-1,1
4991,2026-04-16 10:32:00,1.17757,-0.000140,-1,1
4992,2026-04-16 10:34:00,1.17746,-0.000241,-1,1
4993,2026-04-16 10:36:00,1.17747,-0.000222,-1,1
4994,2026-04-16 10:38:00,1.17750,-0.000182,-1,1
4995,2026-04-16 10:40:00,1.17757,-0.000100,-1,0
4996,2026-04-16 10:42:00,1.17768,0.000023,-1,0
4997,2026-04-16 10:44:00,1.17766,0.000015,-1,0
4998,2026-04-16 10:46:00,1.17773,0.000098,-1,0
4999,2026-04-16 10:48:00,1.17773,0.000111,-1,0


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Define your Features (X) and your Target (y)
# We use the features we created in the previous step
features = ['dist_from_sma_15m', 'trend_4h', 'tick_volume', 'spread']
X = final_df[features]
y = final_df['target']

# 2. Split into Training and Testing sets
# We use 80% of the data to train and 20% to test if the brain actually learned
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Initialize and Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Check the Results
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Training Complete!")
print(f"Accuracy Score: {accuracy:.2%}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

Model Training Complete!
Accuracy Score: 54.36%

Detailed Report:
              precision    recall  f1-score   support

           0       0.56      0.56      0.56       522
           1       0.52      0.52      0.52       475

    accuracy                           0.54       997
   macro avg       0.54      0.54      0.54       997
weighted avg       0.54      0.54      0.54       997



In [4]:
import joblib

# Save the brain to a file
joblib.dump(model, '../models/trading_model_v1.pkl')
print("Model saved to python/models/trading_model_v1.pkl")

Model saved to python/models/trading_model_v1.pkl
